# Radio Bulletin Generator — Three Languages

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Generate natural, radio-ready weather bulletins in **three languages**:
1. **English** (Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

- **Target length**: ~750 words (~5 minutes at 150 wpm radio reading pace)
- **Style**: Philippine radio broadcast — authoritative, clear, flowing prose
- **Model**: Gemma 4 E4B via Ollama (text-only — no vision needed here)
- **Input**: Full markdown bulletin text (from notebook 04)
- **Output**: Plain text scripts saved to `data/radio_bulletins/`

### Two sample bulletins
1. `PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito
2. `PAGASA_22-TC02_Basyang_TCA#01` — Tropical Cyclone Alert, Basyang

**Total output**: 6 scripts (2 bulletins × 3 languages)

In [1]:
import json
import requests
import time
from pathlib import Path

OLLAMA_API = "http://localhost:11434"
MODEL_NAME = "gemma4:e4b"
TIMEOUT = 10 * 60  # 10 minutes

data_dir = Path("../data")
markdown_dir = data_dir / "gemma4_results"
output_dir = data_dir / "radio_bulletins"
output_dir.mkdir(exist_ok=True)

# The two bulletins we are working with
SAMPLES = [
    "PAGASA_20-19W_Pepito_SWB#01",
    "PAGASA_22-TC02_Basyang_TCA#01",
]

# Verify Ollama is running
try:
    r = requests.get(f"{OLLAMA_API}/api/tags", timeout=5)
    names = [m['name'] for m in r.json()['models']]
    status = "\u2713" if any(MODEL_NAME in n for n in names) else "\u26a0\ufe0f  model not found"
    print(f"\u2713 Ollama running \u2014 {MODEL_NAME}: {status}")
except Exception as e:
    print(f"\u2717 Ollama not reachable: {e}")

print(f"\u2713 Output dir: {output_dir.absolute()}")


✓ Ollama running — gemma4:e4b: ✓
✓ Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins


## 1. Load Bulletin Markdown

Load the raw markdown text extracted by notebook 04. This is the primary input — no structured JSON used.

In [2]:
def load_bulletin(stem: str) -> dict:
    """Load the raw markdown for a bulletin stem (primary input for generation)."""
    markdown_path = markdown_dir / f"{stem}_markdown.md"

    if not markdown_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {markdown_path}")

    markdown = markdown_path.read_text(encoding="utf-8")

    return {"stem": stem, "markdown": markdown}


bulletins = [load_bulletin(s) for s in SAMPLES]

for b in bulletins:
    print(f"\n{b['stem']}")
    print(f"  Markdown: {len(b['markdown'])} chars")
    print(f"  Preview:  {b['markdown'][:120].strip()!r}")



PAGASA_20-19W_Pepito_SWB#01
  Markdown: 4825 chars
  Preview:  '## REPUBLIC OF THE PHILIPPINES\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PHILIPPINE ASTROPHYSICAL, GEOPHYSICAL AND ASTR'

PAGASA_22-TC02_Basyang_TCA#01
  Markdown: 3224 chars
  Preview:  '# PHILIPPINE ARCHIPELAGO\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PAGASA**\nPhilippine Atmospheric, Geophysical and Ast'


## 2. Radio Bulletin Prompts — Three Languages

Generate radio bulletins in:
1. **English** (standard Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

All versions maintain the same style:
- Flowing prose — no bullet points, no tables, no markdown

- ~750 words (~5 minutes reading time)- Clear structure: opening → current situation → forecast → affected areas → public safety → closing

- Repeat storm name and signal areas (radio listeners may tune in mid-broadcast)- Plain spoken numbers

In [3]:
PROMPTS = {
    "en": {
        "system": """You are a Philippine radio weather broadcaster writing a spoken bulletin script for on-air reading in English.

STYLE RULES:
- Write in flowing, natural prose suitable for reading aloud.
- Use a calm, authoritative tone appropriate for public emergency broadcasts.
- Spell out numbers when they will be read aloud (e.g. "one hundred thirty kilometres per hour", "Signal Number Two").
- Repeat the storm name and the most critical warning signal at least twice — radio listeners may tune in partway through.
- Use natural spoken transitions: "At this time...", "Moving on to the forecast...", "Residents are urged to..."
- Reference the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA, by name at the start.
- Close with the time of the next scheduled bulletin update.

FORMATTING: Output in Markdown so the script is easy to read and review.
- Use a top-level heading for the bulletin title (storm name + bulletin type)
- Use second-level headings for each section: Current Situation, Forecast Track, Affected Areas, Public Safety Advisory, Closing
- Bold the storm name and signal numbers on first mention in each section
- Keep the prose itself natural and speakable — the markdown is for readability only

LENGTH: Target approximately 750 words — this should read aloud in about five minutes at a steady broadcast pace.""",

        "user_template": (
            "Write a five-minute radio broadcast bulletin script in English "
            "based on the following PAGASA weather bulletin text.\n\n"
            "{markdown}\n\n"
            "Write the radio script now. Use Markdown formatting, approximately 750 words."
        ),
    },

    "tl": {
        "system": """Ikaw ay isang Filipino radio weather broadcaster na sumusulat ng spoken bulletin script para basahin sa ere sa Tagalog.

MGA PATAKARAN SA ESTILO:
- Sumulat sa natural na daloy ng prosa na angkop para basahin nang malakas.
- Gumamit ng kalmado at may awtoridad na tono na angkop para sa public emergency broadcasts.
- I-spell out ang mga numero kapag babasahin (halimbawa: "isandaan at tatlumpung kilometro bawat oras", "Signal Number Two").
- Ulitin ang pangalan ng bagyo at ang pinaka-kritikal na babala nang kahit dalawang beses.
- Gumamit ng natural na transisyon: "Sa ngayon...", "Patungo sa forecast...", "Hinihikayat ang mga residente na..."
- Banggitin ang PAGASA sa simula ng bulletin.
- Magtapos sa oras ng susunod na scheduled bulletin update.

FORMATTING: Mag-output sa Markdown para madaling basahin at suriin.
- Gumamit ng top-level heading para sa pamagat ng bulletin (pangalan ng bagyo + uri ng bulletin)
- Gumamit ng second-level headings para sa bawat seksyon: Kasalukuyang Sitwasyon, Forecast Track, Mga Apektadong Lugar, Payo sa Kaligtasan, Pangwakas
- I-bold ang pangalan ng bagyo at signal numbers sa unang pagbanggit sa bawat seksyon
- Panatilihing natural ang prosa para mabasa nang maayos

HABA: Target na humigit-kumulang 750 salita — dapat itong mabasa sa loob ng limang minuto.""",

        "user_template": (
            "Sumulat ng limang minutong radio broadcast bulletin script sa Tagalog "
            "batay sa sumusunod na PAGASA weather bulletin text.\n\n"
            "{markdown}\n\n"
            "Sumulat ng radio script ngayon. Gumamit ng Markdown formatting, humigit-kumulang 750 salita."
        ),
    },

    "ceb": {
        "system": """Ikaw usa ka Filipino radio weather broadcaster nga direkta nakigsulti sa mga mamumuno sa Cebuano. Isulat ang bulletin sama sa imong gisulti sa radyo — natural, conversational, ug dali masabtan.

MGA LAGDA SA ESTILO (CONVERSATIONAL RADIO CEBUANO):
- Pagsulat sama sa regular nga radyo broadcaster nga nakigsulti sa mga tagapaminaw — dili formal, natural lang nga Bisaya.
- Gamita ang mga common nga radio phrases: "Ato pang hisgutan...", "Karon, mga higala...", "Unya ha...", "Importante nga..."
- I-spell out ang mga numero sama sa pagsulti (pananglitan: "usa ka gatos ug katluan ka kilometro matag oras", "Signal Number Two").
- Sublia ang ngalan sa bagyo ug signal areas og duha o tulo ka beses — basin lang dili makahibalo ang uban.
- Gamita ang conversational nga transitions: "Karon ha...", "Unya kini...", "Importante kaayo nga...", "Mga kauban..."
- Mention PAGASA sa sinugdan pero dili kaayo formal — natural lang.
- Tapuson sama sa regular nga radyo: "Tan-awa nato pag-usab sa..." o "Magkita tag usab sa..."

FORMATTING: Mag-output sa Markdown aron dali mabasahan.
- Paggamit og top-level heading alang sa titulo sa bulletin (ngalan sa bagyo + matang sa bulletin)
- Paggamit og second-level headings alang sa matag seksyon pero simple lang: Unsay Nahitabo Karon, Asa Padulong ang Bagyo, Kinsa ang Apektado, Unsa ang Kinahanglan Buhaton, Pangwakas
- I-bold ang ngalan sa bagyo ug signal numbers
- Pero ang script mismo kinahanglan conversational — parang nag-istambay lang mo sa radyo

GITAS-ON: Mga 750 ka pulong — lima ka minuto nga radyo talk, dili basahon nga formal.""",

        "user_template": (
            "Isulat ang lima ka minutong conversational radio bulletin sa Cebuano "
            "base sa PAGASA weather bulletin text nga naay sulod dinhi.\n\n"
            "{markdown}\n\n"
            "Pagsulat karon — conversational Bisaya, parang nag-istorya lang sa radyo. Markdown format, mga 750 ka pulong."
        ),
    },
}


def build_user_prompt(bulletin: dict, language: str) -> str:
    """Build the user prompt using the full markdown bulletin text as input."""
    template = PROMPTS[language]["user_template"]
    return template.format(markdown=bulletin["markdown"])


print("\u2713 Prompts defined for 3 languages")
for lang, prompts in PROMPTS.items():
    print(f"  {lang}: system={len(prompts['system'])} chars, template={len(prompts['user_template'])} chars")


✓ Prompts defined for 3 languages
  en: system=1362 chars, template=206 chars
  tl: system=1304 chars, template=227 chars
  ceb: system=1579 chars, template=250 chars


## 3. Generate Radio Bulletins — All Three Languages

Call Gemma 4 for each bulletin in English, Tagalog, and Cebuano (6 total scripts).

In [4]:
def generate_radio_bulletin(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a radio broadcast script for one bulletin in the specified language."""
    stem = bulletin["stem"]
    lang_names = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}
    print(f"\nGenerating: {stem} ({lang_names[language]})")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": PROMPTS[language]["system"]},
            {"role": "user", "content": build_user_prompt(bulletin, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    elapsed = time.time() - t_start

    script = response.json().get("message", {}).get("content", "").strip()

    # Word count (strip markdown syntax for accurate spoken-word estimate)
    import re
    plain = re.sub(r"[#*_`>\-]", "", script)
    word_count = len(plain.split())
    reading_minutes = word_count / 150  # ~150 wpm for radio

    # Save as markdown file
    out_path = output_dir / f"{stem}_radio_{language}.md"
    out_path.write_text(script, encoding="utf-8")

    print(f"  \u2713 Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}  |  Est. reading time: {reading_minutes:.1f} min")
    print(f"  Saved \u2192 {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "script": script,
        "word_count": word_count,
        "reading_minutes": reading_minutes,
        "elapsed": elapsed,
    }


results = []
for bulletin in bulletins:
    for lang in ["en", "tl", "ceb"]:
        result = generate_radio_bulletin(bulletin, lang)
        results.append(result)

print(f"\n\u2713 Done \u2014 {len(results)} scripts generated ({len(bulletins)} bulletins \u00d7 3 languages)")



Generating: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 42.0s
  Words: 596  |  Est. reading time: 4.0 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_en.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 100.2s
  Words: 719  |  Est. reading time: 4.8 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_tl.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 89.6s
  Words: 641  |  Est. reading time: 4.3 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_ceb.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 70.9s
  Words: 714  |  Est. reading time: 4.8 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_en.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 91.1s
  Words: 702  |  Est. reading time: 4.7 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_tl.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 91.6s
  Words: 662  |  Est. reading time: 4.4 min
  Saved → PAGASA_22-TC02_Basya

## 6. TTS Plain Text Generation — Cebuano

Generate a TTS-optimized plain text version of each Cebuano radio script via a second
Gemma4 prompt. This file feeds directly into notebook 08 — no markdown stripping needed.

**Output:** `data/radio_bulletins/{stem}_tts_ceb.txt`

In [ ]:
TTS_PROMPTS = {
    "ceb": {
        "system": """Ikaw usa ka espesyalista sa Cebuano nga nagsulat og plain text nga angay para sa text-to-speech synthesis.

Ang imong trabaho:
- Basaha ang markdown radio script nga gihatag
- Isulat kini pag-usab isip natural nga flowing prose SA CEBUANO LAMANG — walay markdown
- WALA markdown: wala headings (#), wala bullet points (-), wala asterisks (*), wala bold/italic
- Para sa mga English proper nouns o teknikal nga termino, i-spell sila phonetically sa Cebuano:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tros sa usa ka oras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Pahimusa ang paragraph structure: blank lines tali sa mga paragraph
- AYAW pagdugang og bisan unsa nga texto nga wala sa orihinal nga script
- Output: plain text lamang, walay bisan unsang markup o formatting characters""",

        "user_template": (
            "Basaha kining markdown radio script ug isulat kini pag-usab isip TTS-ready plain Cebuano text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Cebuano text karon. Cebuano nga pulong lamang, phonetically spelled kung "
            "kinahanglan, paragraph breaks (blank lines) para sa natural nga pausing. Walay markdown."
        ),
    },
}


def build_tts_prompt(markdown_script: str, language: str) -> str:
    """Build the user prompt for TTS text generation."""
    return TTS_PROMPTS[language]["user_template"].format(markdown=markdown_script)


print("✓ TTS_PROMPTS defined for CEB")
print(f"  ceb: system={len(TTS_PROMPTS['ceb']['system'])} chars")

In [ ]:
def generate_tts_text(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a TTS-ready plain text script for one bulletin."""
    stem = bulletin["stem"]
    lang_names = {"ceb": "Cebuano"}
    print(f"\nGenerating TTS text: {stem} ({lang_names[language]})")

    markdown_script = (output_dir / f"{stem}_radio_{language}.md").read_text(encoding="utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": TTS_PROMPTS[language]["system"]},
            {"role": "user", "content": build_tts_prompt(markdown_script, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    elapsed = time.time() - t_start

    tts_text = response.json().get("message", {}).get("content", "").strip()

    out_path = output_dir / f"{stem}_tts_{language}.txt"
    out_path.write_text(tts_text, encoding="utf-8")

    word_count = len(tts_text.split())
    print(f"  ✓ Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}")
    print(f"  Saved → {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "tts_text": tts_text,
        "word_count": word_count,
        "elapsed": elapsed,
    }


# Generate TTS text for both bulletins — CEB only
tts_results = []
for bulletin in bulletins:
    result = generate_tts_text(bulletin, "ceb")
    tts_results.append(result)

print(f"\n✓ Done — {len(tts_results)} TTS text files generated")

# Preview first bulletin
print("\nPREVIEW — CEB TTS text (first 500 chars)")
print("=" * 60)
print(tts_results[0]["tts_text"][:500])
print("...")

## 4. Review Output

Display each generated script for manual inspection (grouped by bulletin, showing all languages).

In [5]:
from IPython.display import display, Markdown
from collections import defaultdict

grouped = defaultdict(list)
for r in results:
    grouped[r["stem"]].append(r)

lang_labels = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}

for stem, versions in grouped.items():
    display(Markdown(f"---\n# {stem}\n---"))
    for version in versions:
        lang = version["language"]
        meta = (
            f"**Language:** {lang_labels[lang]}  \u2502  "
            f"**Words:** {version['word_count']}  \u2502  "
            f"**Est. reading time:** {version['reading_minutes']:.1f} min"
        )
        display(Markdown(f"## {lang_labels[lang]} Version\n\n{meta}\n\n---\n\n{version['script']}"))


---
# PAGASA_20-19W_Pepito_SWB#01
---

## English Version

**Language:** English  │  **Words:** 596  │  **Est. reading time:** 4.0 min

---

# Tropical Depression "PEPITO" - Severe Weather Advisory Bulletin

## Current Situation

Good morning. This is your weather advisory bulletin brought to you by the **Philippine Atmospheric, Geophysical and Astronomical Services Administration**, or PAGASA. We are issuing this bulletin regarding **Tropical Depression "PEPITO."**

As of this time, **Tropical Depression "PEPITO"** has developed from a low-pressure area situated east of Catanduanes. PAGASA meteorologists are closely monitoring its movement and intensity. At the moment of this broadcast, while the storm has been classified as a **Tropical Depression**, there is a chance that it may further intensify into a Tropical Storm or even a Typhoon as it approaches landmasses.

Residents should remain vigilant, as the system continues to pose a threat. We repeat: we are monitoring **Tropical Depression "PEPITO,"** and please remain alert for potential changes in its signal number.

## Forecast Track

Moving on to the forecast track, PAGASA projects that **Tropical Depression "PEPITO"** is expected to move in a general west-northwestward or north-northwestward direction today. Over the next few days, the storm is forecast to turn more decidedly westward, eventually aiming for Northern Luzon.

Looking ahead, the forecast indicates that **Tropical Depression "PEPITO"** is most likely to make landfall over the eastern coast of Northern Luzon during the afternoon of October twenty-first.

The predicted track shows the system traversing the waters near Catanduanes, heading toward the eastern coasts of Samar and Leyte, before curving northwestward across the waters near Northern Luzon. The intensity at the center is currently estimated at sustained winds of up to forty-five kilometers per hour, with gusts potentially reaching fifty-five kilometers per hour.

## Affected Areas

The areas expected to be most severely affected by **Tropical Depression "PEPITO"** are those along the eastern coasts. Specifically, we are advising heightened caution for the provinces of Catanduanes, Northern Samar, Samar, and Southern Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, and Quezon.

Furthermore, coastal residents in Northern Luzon, including Isabela, Cagayan, Ilocos Norte, Aurora, Zambales, Bataan, and parts of Pangasinan, should also prepare for impacts. The Bicol Region, as well as the aforementioned provinces, can expect a combination of strong winds and significant rainfall.

At this time, PAGASA has warned that the seas in the northeastern to former southern side of the region are expected to be moderate to strong, with wave heights potentially reaching between two point five and four point five meters. Extreme caution is necessary for anyone utilizing small sea craft or commercial vessels, and waterways are expected to be particularly hazardous.

## Public Safety Advisory

Residents are urged to take proactive measures immediately. Given the potential for heavy rainfall, especially tonight, we advise all households to secure loose objects outdoors and to check emergency kits.

For coastal communities, please heed advisories from your local disaster risk reduction and management council. Do not attempt to cross rivers or travel by sea if the conditions are rough. PAGASA advises that while no Tropical Cyclone Wind Signal is currently in effect, residents must be prepared to react quickly should the signal level be raised.

We emphasize that **Tropical Depression "PEPITO"** poses a significant hazard, and continuous monitoring is crucial. We must remain prepared for rapid changes.

## Closing

In summary, **Tropical Depression "PEPITO"** is moving westward, with landfall projected for the eastern coast of Northern Luzon. We ask everyone to stay tuned to this frequency for updates from PAGASA.

We will continue to monitor the situation minute by minute and will issue the next Severe Weather Bulletin update at eleven o’clock in the morning today. For more information, please visit the official PAGASA website. Stay safe, and may God bless you all.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 719  │  **Est. reading time:** 4.8 min

---

# TROPICAL DEPRESSION “PEPITO”: Severe Weather Bulletin Advisory

**(Sound Effect: Short, authoritative musical jingle, fades quickly)**

**Boses:** Magandang umaga po sa ating lahat. Ito po ay isang pag-uulat mula sa PAGASA, o Philippine Atmospheric, Geophysical and Astronomical Services Administration. Narito ang inyong Severe Weather Bulletin, na inihahatid sa inyo ngayon.

Ang ating pag-uulat ay patungkol sa pag-usad ng Tropical Depression na **PEPITO**. Ang paghahanda at pagiging handa ang susi upang mapagaan ang epekto ng bagyong ito. Pakikinggan po ninyo ang mga detalye nang mabuti.

***

### Kasalukuyang Sitwasyon

Sa kasalukuyan, isang Low Pressure Area na matatagpuan sa silangan ng Catanduanes ang nag-develop na at pormal nang tinawag na Tropical Depression na **PEPITO**. Ang **PEPITO** ay isang sistema na patuloy na binabantayan ng mga awtoridad sa panahon dahil sa potensyal nitong lumakas pa.

Ayon sa mga datos mula sa PAGASA, ang **PEPITO** ay may kakayahang lumakas pa patungo sa Tropical Storm, o kaya naman ay maging isang malakas na bagyo. Sa kasalukuyan, ang inaasahang lakas ng hangin ay nasa hanggang apatnapu't limang kilometro bawat oras, na may mga pag-ihip na umaabot hanggang limampu't limang kilometro bawat oras.

Ang mga babala po sa dagat ay mahalaga ring idiin: Inaasahan ang pagtaas ng alon at pagbaha sa mga baybayin. Ang karagatan ay magiging katamtaman hanggang malakas, na may mga bahagi na umaabot sa dalawang kuwarter at limang metro hanggang apat na kuwarter at limang metro.

***

### Forecast Track

Patungkol naman sa tinataya nating paggalaw ng **PEPITO**, ang tinatarget nitong direksyon ay pa-kanluran patungong hilaga-kanluran, bago ito lalo pang lumiko patungong hilagang bahagi ng Luzon.

Tandaan po natin ang mga kritikal na prediksyon: Inaasahang lalapit ang bagyo sa baybayin ng hilagang Luzon pagdating ng hapon ng dalawampu't isa (21) ng Oktubre.

Sa paglipas ng mga araw, ipinakita ng ating forecast track na ang **PEPITO** ay dadaan sa karagatan sa silangan ng Catanduanes, at malapit sa baybayin ng Samar at Leyte. Patuloy itong liliko patungong hilaga-kanluran, at ipinakita ring maaaring dumaan ito sa mga bahagi ng Pangasinan at Isabela.

Ang pag-iingat po sa paggalaw ng hangin at ulan ay hindi dapat ipagwalang-bahala, lalo na sa mga paglipas ng araw.

***

### Mga Apektadong Lugar

Ang mga rehiyon na inaasahang magiging pinaka-apektado ng **PEPITO** ay ang mga sumusunod:

Una po, ang malalawak na bahagi ng Visayas, partikular ang Catanduanes, Northern Samar, Samar, at Southern Leyte. Ito po ang mga lugar na inaasahang mararanasan ang matitinding pagbuhos ng ulan ngayong gabi.

Ikalawa naman, ang mga katabing bahagi ng Luzon. Kabilang dito ang Bicol Region, Quezon, Masbate, Albay, at iba pang mga kabayanan sa hilagang bahagi.

Ang mga baybayin na matatagpuan sa mga probinsya ng Ilocos Norte, Aurora, Zambales, at Bataan ay dapat ring maging alerto. Ang lahat ng mga komunidad na malapit sa baybayin ay dapat maghanda sa matitinding paghampas ng hangin at pagbaha.

***

### Payo sa Kaligtasan

Hinihikayat po ng PAGASA ang lahat ng residente na maging alerto at gumawa ng mga hakbang upang maprotektahan ang inyong pamilya at ari-arian.

Una, pangalagaan ang inyong mga buhay. Huwag pong mag-abang ng mga bangka o anumang sasakyang tubig malapit sa mga baybayin hangga't hindi nagwawakas ang babala.

Pangalawa, maghanda ng mga emergency kit. Siguraduhin na mayroon kayong sapat na supply ng tubig, pagkain, at mga pangunahing gamot.

At pangatlo, tandaan po natin ang pangunahing babala: Ang patuloy na pagbabantay sa paglakas ng hangin at malakas na pagbuhos ng ulan ay kritikal. Paulit-uli po nating babalikan ang mahalagang babala na **ang paghahanda para sa matinding ulan at malalakas na hangin na dala ng Tropical Depression na PEPITO ay dapat nating pangunahing prayoridad**.

Ang mga lokal na pamahalaan ay inaasahang magsasagawa ng mga paglikas sa mga lugar na itinuturing na pinakamapanganib. Makinig po kayo sa inyong mga opisyal na awtoridad.

***

### Pangwakas

Mga kababayan, ang kaligtasan po ay higit sa lahat. Kahit pa wala pa pong signal na itinaas, hindi po dapat manawagan ng pag-iingat. Ang pagiging kalmado at pagiging maagap ang magdadala sa atin sa ligtas na pagdaan sa epekto ng bagyo.

Muli po nating ipinaaalala, ang paghahanda para sa matinding ulan at malalakas na hangin na dala ng Tropical Depression na **PEPITO** ay napakahalaga. Manatiling kalmado, at sumunod sa mga paalala ng inyong lokal na pamahalaan.

Ito po ang inyong pag-uulat mula sa istasyon ng panahon. Muli po kaming magpapaalala na manatiling kaligtas, at mag-iingat po kayo sa darating na mga oras. Maraming salamat po.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 641  │  **Est. reading time:** 4.3 min

---

# **TROPICAL DEPRESSION "PEPITO" WEATHER ALERT: UPDATE SA BAGYO**

**(Sound effect: Short, dramatic news stinger music fades out slowly)**

**BROADCASTER:** Maayong buntag kaninyong tanan! Karon, mga higala, pag-andam na mo kay usa ka importanteng update gikan sa mga eksperto sa PAGASA—sa Philippine Atmospheric, Geophysical and Astronomical Services Administration. Ang paghisgot nato karon kay bahin sa usa ka tropical depression nga gitawag og **PEPITO**.

Ato pang hisgutan pag-ayo kay dili ni basta-basta nga weather update. Kinahanglan kaayo ninyo pagtagad aning mga detalye. Pagpahinumdom lang, kami maghatag ni og update, pero ang kaligtasan ninyo, mga kauban, mao ang pinakaimportante.

***

### **Unsay Nahitabo Karon?**

**BROADCASTER:** Sa pagmata nato ning adlawa, base pa sa gibugkos sa PAGASA, ang usa ka Low Pressure Area nga atubangan sa Catanduanes, nag-develop na og Tropical Depression, nga mao ra gyud ni ang **PEPITO**.

Karon, mga higala, ang **PEPITO** adunay kasughanan nga mahimong mo-intensify pa—pwede pa kining mahimong Tropical Storm o, bisan pa, usa ka Tropical Cyclone. Mao ni ang atong pangandam nga padayonon sa pagpagawas og mga warning.

Importante nga dili mo-panic, pero kinahanglan gyud mo kaandam. Kung naa mo’y mga plano nga mo-travel, ilabi na sa mga coastal areas, ayaw gyud ninyo kining pasagdi.

***

### **Asa Padulong ang Bagyo?**

**BROADCASTER:** Karon, adto na ta sa pangutana nga "Asa man ni padulong?"

Ang **PEPITO** kay mo-move paingon sa kabilin-an, mga kauban. Ang atong forecast nag-ingon nga mo-west-northwestward o north-northwestward kini karon. Paglabay sa adlaw, mag-shift siya og direksyon ug mas mo-westward pa, padulong pa sa Northern Luzon.

Ug atong pagtagad, gi-forecast nga ang **PEPITO** makadangat o mo-landfall sa eastern coast sa Northern Luzon sa hapon sa adlaw nga Octobre ika-kalabaynobe. Mao kana ang importante nga appointment sa atong pagtagad.

Ang kasamtangang kusog niini? Ang mga uban makakita og wind gusts nga moabot pa sa 55 kilometro sa kada oras. Bisan og wala pa'y signal, dili pa man kini nagpasabot nga luwas na mo.

***

### **Kinsa ang Apektado ug Unsa’y Panglimbong?**

**BROADCASTER:** Karon na, mga higala, ang pinaka-kritikal nga bahin—kinsa ba ang magpabati og grabe nga epekto aning **PEPITO**?

Dili lang ang mga coastal areas, pero kinahanglan ninyo ning makatala:

1.  **Ang Visayas Region:** Ang forecast nag-ingon nga ang Visayas region, ilabi na pag-esta sa gabii, mag-atubang og **bug-at nga pag-ulan**. Mao ni nga kinahanglan gyud ninyo nga pag-andam sa pag-ulan.
2.  **Ang Bicol Region ug Catanduanes:** Kining mga probinsya kay nag-expect og ulan ug kusog nga hangin.
3.  **Mga Probinsya sa Samar, Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, ug Quezon:** Kini nga mga area, mga kauban, gihisgotan nga **pinaka-apektado** sa pag-abot sa **PEPITO**. Pag-andam pag-ayo kinyo.
4.  **Northern Luzon:** Kining mga probinsya sa Ilocos Norte, Aurora, Cagayan, Zambales, ug Bataan, dili pud malikayan, kay mo-pass-by ra gihapon kining mga hangin ug ulan.

**BROADCASTER:** Ug kung mo-travel mo sa mga dagat, mga igsoon, ang mga signal sa dagat kay moderate to strong, nga moabot pa sa duha ka ka-kwartos ug lima ka mga metro! Kung mo-operate mo sa bangka o commercial vessel, pag-amping gyud!

***

### **Unsa ang Kinahanglan Buhaton?**

**BROADCASTER:** Mga kauban, daghan kaayong giingon, pero unsa gyud ang ato buhaton? Simple ra, pero kinahanglan gyud ninyo ning sundon:

Unang-una, **Paghandaon ang inyong evacuation kit**. Pag-check sa inyong mga emergency supplies. Tubig, pagkaon nga dali kaonon, first aid kit.

Ikaduha, **Pag-monitor sa mga opisyal nga mga channel.** Ayaw pag-salig sa mga tambayayong nga mga istorya sa social media. Kanunay nga pag-monitor sa mga broadcast sa PAGASA, mga lokal nga pamahalaan, ug sa mga awtoridad sa inyong lugar.

Ikatulo, **Para sa mga nag-uwi o mo-travel:** Kung ang mga signal sa hangin ug ulan kay grabe, ang tambag kay *i-delay ang pag-biyahe*. Pag-stay safe sa balay.

Ang PAGASA nag-ingon nga **WALAY** Tropical Cyclone Wind Signal nga nag-effect karon. Pero pag-andam ta kay kanus-a man mo-signal.

**Summary lang: Stay updated, stay safe, and prepare for the worst.**

Mag-amping mo tanan. Magkita-kita ta pag-ayo nga safe na tanan!

---
# PAGASA_22-TC02_Basyang_TCA#01
---

## English Version

**Language:** English  │  **Words:** 714  │  **Est. reading time:** 4.8 min

---

# Tropical Storm Malakas: Weather Bulletin and Advisory

## Current Situation
Good morning, this is your official weather update brought to you by the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA. We are broadcasting the latest advisory regarding **Tropical Storm Malakas**.

At this time, the center of **Tropical Storm Malakas** was last estimated to be located approximately two point zero zero kilometers east of Mindanao. We are monitoring this system closely from our headquarters here in Quezon City.

As of our last assessment, **Tropical Storm Malakas** maintained maximum sustained winds of seventy-five kilometres per hour, with gustiness recorded up to ninety kilometres per hour. The current movement of the storm center is reported at a steady pace of fifteen kilometres per hour, tracking generally in a west-northwest direction.

While the storm remains outside of our immediate Philippine Area of Responsibility, it is crucial that everyone remain vigilant. PAGASA continues to gather data from multiple sources, and we urge all residents to stay tuned to local radio stations and follow the official advisories from PAGASA for the most accurate and timely information.

## Forecast Track
Moving on to the forecast track, the trajectory of **Tropical Storm Malakas** shows a continuing path westward.

Over the next twenty-four hours, the storm is forecast to track approximately one thousand, six hundred twenty kilometres east of Mindanao. As the system moves, we are observing an increasing pattern of intensity.

Looking further ahead, by the thirty-six-hour mark, **Tropical Storm Malakas** is expected to be around one thousand, four hundred forty kilometres east of the Visayas region. The model projections indicate that the system has the potential to intensify further, reaching peak forecasted sustained winds of one hundred and ten kilometres per hour by that point.

For those tracking this system through the next several days, the forecasts indicate a gradual but steady strengthening trend. We see projections reaching over one hundred and fifty kilometres per hour, and further out, the system is forecasted to reach Typhoon strength. We repeat: the potential intensity for this system, **Tropical Storm Malakas**, is tracking towards Typhoon intensity across the board.

These projections are critical, as they illustrate the significant power building within the storm complex, even as it remains offshore. We must prepare for the possibility of an elevated threat level, which is why preparedness is paramount right now.

## Affected Areas
Regarding the specific areas affected, it is important to note that PAGASA currently forecasts **Tropical Storm Malakas** to remain outside of the Philippine Area of Responsibility for the majority of the upcoming period. This means that the most direct impacts of the storm's center are not expected to hit any populated areas in the immediate term.

However, we must caution the public that even offshore tropical cyclones can generate rough seas, strong winds, and associated adverse weather conditions far from the center.

Residents in areas prone to storm surges, coastal flooding, or high winds from approaching tropical systems are strongly advised to review your local evacuation plans immediately. Do not wait for an official warning before making necessary preparations.

## Public Safety Advisory
Residents are urged to take proactive measures before the weather situation escalates.

First, secure all loose outdoor objects—such as roofing materials, patio furniture, and signage—that could become projectiles in strong winds.

Second, review your family's emergency kits. Ensure you have at least three days' worth of non-perishable food, clean drinking water, necessary medications, and portable lighting sources.

Third, and most importantly, please remain updated. Pay close attention to any changes in the wind signals or any advisories indicating that **Tropical Storm Malakas** has entered the Philippine Area of Responsibility.

Remember, the resilience of our communities is our greatest defense. Stay informed, stay safe, and heed all instructions issued by local government units and PAGASA personnel.

## Closing
This has been your comprehensive weather advisory from PAGASA. We understand that weather advisories can cause anxiety, but please remember that preparation is our best defense.

We will continue to monitor **Tropical Storm Malakas** with the highest degree of vigilance and will issue an updated bulletin every three hours, or immediately if there are significant changes to its track or intensity. Our next scheduled bulletin update will air at eleven o'clock in the morning. Stay safe, and keep listening to your local stations for further updates.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 702  │  **Est. reading time:** 4.7 min

---

# **Tropical Storm MALAKAS Advisory: Updated Public Safety Bulletin**

**(SOUND EFFECT: Pag-ingat na chime o maikling, kalmadong background music)**

**BROADCASTER:** Magandang umaga po. Ito ang inyong weather bulletin mula sa PAGASA.

**PAGASA:** Sa ngalan ng kapayapaan at kaligtasan, narito po ang inyong pag-uulat ng panahon mula sa Philippine Atmospheric, Geophysical and Astronomical Services Administration.

Hinihiling po namin sa lahat ng ating mga kababayan na manatiling kalmado at alerto. Sa ngayon, mayroon tayong pag-uulat tungkol sa paggalaw at pagpapatibay ng isang tropical cyclone, ang **Tropical Storm MALAKAS**. Ito po ay isang sitwasyong nangangailangan ng ating buong pag-iingat.

---

## **Kasalukuyang Sitwasyon**

Sa kasalukuyan, ang gitna ng **Tropical Storm MALAKAS** ay naitala sa lapit ng Mindanao, ngunit nasa labas po pa rin ng ating Philippine Area of Responsibility, o PAR. Base sa ating pag-aaral kaninang madaling araw, ang sentro ng **Tropical Storm MALAKAS** ay matatagpuan sa tinatayang dalawang po (2) punto (2) kilometro silang (km) silang, silangan ng Mindanao.

Ang lakas nito po ay kasalukuyan pang malakas. Ayon sa datos, ang pinakamataas na paghawak na hangin ay aabot sa pitumpu’s (70) kilometro bawat oras, na may mga pagbuga na posibleng umabot sa siyamnapu (90) kilometro bawat oras. Patuloy itong nagpapakita ng pagpapatibay.

Ang paggalaw nito ay nasa direksyong kanluran-hilagang-kanluran. Ang pangunahing babala po na nais naming ipabatid ay ang **pagpapatuloy ng pagpapatibay at paglipat sa mga rehiyon sa tabi ng PAR**. Ito po ay nangangahulugang patuloy nating babantayan ang pag-akyat ng intensity nito.

---

## **Forecast Track at Pagbabago ng Intensity**

Patungo po tayo ngayon sa forecast o inaasahang paggalaw ng bagyo.

Ang datos mula sa PAGASA ay nagpapakita na habang nagpapatuloy ang paggalaw ng **Tropical Storm MALAKAS** sa direksyong kanluran-hilagang-kanluran, patuloy itong nagpapakita ng pagpapatibay.

Kung susundan natin ang pag-uulat sa loob ng nakalipas na dalawampu’sing (20) oras, inaasahang patuloy itong lalakas at aabot sa kategorya na Typhoon o malakas na bagyo. Sa paglipas ng oras, patuloy po itong lalayo sa ating mga pangunahing isla, ngunit dahil sa pagtaas ng intensity, nananatiling kritikal ang paghahanda.

Sa kabuuan, ang pag-uulat ay nagpapahiwatig na ang **Tropical Storm MALAKAS** ay magiging isang malakas na bagyo o isang bagyo na may hangin na posibleng umabot sa isang daan at limampu’sing (150) kilometro bawat oras sa darating na araw.

Ibig sabihin po nito, habang malayo pa, ang paghahanda ay dapat nang simulan. Ang pag-iingat ay hindi lamang para sa araw na ito, kundi para sa mga susunod na araw din.

---

## **Mga Apektadong Lugar at Paghahanda**

Dahil po na ang sentro ng **Tropical Storm MALAKAS** ay nasa labas po ng ating PAR, sa ngayon ay walang direktang babala ng malakas na hangin sa anumang rehiyon.

Gayunpaman, hinihiling po namin sa lahat ng mga residente na naninirahan sa baybayin o sa mga lugar na kadalasang tinatamaan ng malalakas na bagyo, na manatiling lubos na handa.

**Mga Pangunahing Dapat Gawin:**

Una po, ihanda ang inyong mga pamilya. Siguraduhin na mayroon kayong sapat na supply ng inuming tubig, pagkain, at first aid kit.
Pangalawa, alamin ang inyong mga evacuation route. Huwag po kayong mag-atubiling umalis sa inyong lugar kung utusan kayo ng mga awtoridad.
Pangatlo, patuloy na makinig sa mga ulat ng PAGASA at sa inyong Local Government Unit.

---

## **Payo sa Kaligtasan at Pangwakas**

Mga kababayan, tandaan po ninyo: ang paghahanda ay susi sa ating kaligtasan. Hindi po dapat magpabaya dahil paalala lang po na malayo pa ang bagyo. Ang pagpapatibay ng **Tropical Storm MALAKAS** ay nangangahulugan na ang posibleng lakas na maipaparating nito ay malaki pa rin.

Hinihikayat ang lahat na manatiling nasa loob ng inyong mga tahanan hanggang sa magbigay ng opisyal na pag-uulat na ligtas na kayo. Iwasan po ang paglalakbay, lalo na sa mga lugar na mababaha o may landslide risk.

Paalala po na ito ay isang patuloy na pagsubaybay. Mananatiling nakatuon po kami sa pagbibigay sa inyo ng pinakatumpak at pinaka-up-to-date na impormasyon.

Muli, ipinaaalala po namin ang paghahanda para sa **Tropical Storm MALAKAS**. Ang pagiging alerto at pagiging handa ang magliligtas sa ating buhay.

Maaari po kayong makakuha pa ng impormasyon sa opisyal na website ng PAGASA.

At iyan po ang inyong weather bulletin. Sa susunod na pag-uulat po, babalik kami rito sa alas-dose (12:00) ng tanghali. Manatiling ligtas po kayong lahat.

**(SOUND EFFECT: Pag-ingat na chime na nagtatapos sa bulletin)**

## Cebuano Version

**Language:** Cebuano  │  **Words:** 662  │  **Est. reading time:** 4.4 min

---

# Tropical Storm "Malakas" - Weather Update (Radio Broadcast)

**(Sound of gentle, upbeat background music fades in and then fades down to background level)**

**BROADCASTER:** Kumusta na mga kaigsoonan sa Cebu ug sa tibuok visbis? Maayong buntag kaninyong tanan, ug pag-abot ninyo sa atong *Weather Update* aron mapasalamaton gihapon mo sa pagpaminaw sa atong istasyon.

Ato pang hisgutan karon ang pinakabag-o nga sitwasyon sa atong mga ulan ug mga bagyo, nga gi-monitor man gyud sa atong mga ka-eksperto sa PAGASA—ang Philippine Atmospheric, Geophysical and Astronomical Services Administration.

Karon, mga higala, dunay usa ka tropical cyclone nga atong gilantaw, ang gi-tawag og **Tropical Storm "Malakas."** Kini ang topic natong hisgutan sa dili pa mo-uwian ang inyong mga kauban sa trabaho.

***

### Unsay Nahitabo Karon (The Current Situation)

Karon sa pagkakaron, ang **Tropical Storm "Malakas,"** o mas klaro pa, ang **Malakas**, nag-atubang pa gyud sa dagat. Matag oras nga paglabay, kami nag-monitor aning tropikal nga siklon.

Base sa mga pagtahod sa PAGASA, ang **Malakas** nagpabilin gihapon nga lig-on. Ang maximum sustained winds niini giban-aw pa sa pito ka ka, o 75 ka kilometro kada oras. Ang mga hangin diri ka-gamay mo-abot sa siyam ka ka, o 90 ka kilometro kada oras.

Pero, ug importante kaayo ninyong masayran, **malayo pa gyud kaayo** ang atong **Tropical Storm Malakas** sa Philippine Area of Responsibility, o PAR. Ang atong **Malakas** nag-agkun pa gyud sa pangutlo, paingon pa sa kasadlungngan. Busa, sa pagkakaron, ang among mga kauban, kalma lang, pero magpabilin gihapon ta nga alerto.

***

### Asa Padulong ang Bagyo (The Forecast Track)

Karon ha, ato ning atong i-discuss ang atong mga forecast, kay kini ang importante nga bahin.

Ang mga pagtagad sa PAGASA nagpakita nga ang paglihok sa **Tropical Storm Malakas** kay West-Northwest, o pahilayo pa gyud sa atong mga isla.

Mga kauban, kon tan-awon nato ang forecast track, makita nato nga ang **Malakas** nagpadayon sa paglayo. Kini nga bagyo dili pa moabot aron makaapekto kanato direkta. Sa paglabay sa mga sunod nga adlaw, ang **Malakas** magpadayon sa paglihok paglayo, gikan sa atong mga isla.

Kining paglihok nga West-Northwest, nagpasabot nga ang atong **Malakas** nagpadayon sa iyang agianan sa mga tubig nga nagkatingog, sa kasadlungngan sa Mindanao, sa Visayas, hangtud sa mga lugar sa Luzon, pero **wala kini pa'y direkta nga epekto kanato base sa atong forecast karon**. Ang mga numero sa forecast nagpakita nga kini nga bagyo naglayo pa gikan kanato.

***

### Kinsa ang Apektado ug Unsa ang Kinahanglan Buhaton (Safety Advice)

Mga kauban, kanus-a man mo-abot ang bagyo, o kon molig-on pa kini, ang atong pag-andam, kanus-a man nako mag-andam.

Busa, ang mensahe sa PAGASA para sa atong mga kauban mao ang *pagka-andam, dili ang pagka-panic*.

1.  **Paminaw sa Opisyal nga mga Pahibalo:** Importante kaayo nga ang among mga kauban magpadayon sa pagpaminaw sa mga opisyal nga pahibalo—sa PAGASA, sa inyong lokal nga LGU, ug sa atong istasyon. Ayaw mo'g kasaligan sa mga dili opisyal nga mga source sa impormasyon, ha?
2.  **Manalipod sa Pag-uwan ug Hangin:** Bisan pa nga layo pa ang **Malakas**, kinahanglan gihapon nga andam kita sa pag-adto sa taas kon mogawas ang kusog nga hangin o bagyo sa inyong lugar. Siguraduhon nga ang inyong balay, inyong mga mga materyales, ug mga gamit sa sakyanan kay andam sa bisag unsa nga pagbag-o sa panahon.
3.  **Hydration ug Pagkaon:** Mag-andam og saktong suplay sa tubig ug pagkaon. Mao ni'y atong sundon bisan og molungkot pa ang panahon.

***

### Pangwakas (Closing)

Sa kinatibuk-an, mga kaigsoonan, ang pagmonitor sa **Tropical Storm Malakas** nagpadayon. Ang atong pagtuo sa mga datos sa PAGASA nagpakita nga ang **Malakas** naglayo pa gyud sa atong mga isla. Pero, ang pag-amping dili gyud makapahumot.

Magpabilin kita nga magkauyok, mag-amping sa atong pamilya, ug pag-amping sa tanan nga inyong kauban sa pagpaminaw.

Tan-awa nato pag-usab sa mga detalye niining **Tropical Storm Malakas** sa among sunod nga weather update. Bantayi lang ang atong istasyon.

Hangtod nianang pag-usab, mag-amping kamo diha sa inyong mga panimalay, ug daghang salamat sa pagpaminaw. Paalam!

**(Sound of upbeat background music fades up and plays out)**

## 5. Length Check

Verify word counts across all three languages are close to the 750-word target. If any are significantly short or long, we can adjust the prompts.

In [6]:
TARGET_WORDS = 750
WORDS_PER_MIN = 150

print("\nLENGTH SUMMARY")
print("=" * 60)
print(f"{'Bulletin':<35} {'Lang':>4} {'Words':>6}  {'Minutes':>7}  {'vs Target':>10}")
print("-" * 60)

for result in results:
    diff = result['word_count'] - TARGET_WORDS
    diff_str = f"+{diff}" if diff >= 0 else str(diff)
    label = result['stem'].replace('PAGASA_', '')
    lang = result['language'].upper()
    print(f"{label:<35} {lang:>4} {result['word_count']:>6}  {result['reading_minutes']:>6.1f}m  {diff_str:>10}")

print("-" * 60)
print(f"Target: {TARGET_WORDS} words ≈ {TARGET_WORDS/WORDS_PER_MIN:.0f} minutes at {WORDS_PER_MIN} wpm")
print(f"\n✓ {len(results)} scripts saved to: {output_dir.absolute()}")


LENGTH SUMMARY
Bulletin                            Lang  Words  Minutes   vs Target
------------------------------------------------------------
20-19W_Pepito_SWB#01                  EN    596     4.0m        -154
20-19W_Pepito_SWB#01                  TL    719     4.8m         -31
20-19W_Pepito_SWB#01                 CEB    641     4.3m        -109
22-TC02_Basyang_TCA#01                EN    714     4.8m         -36
22-TC02_Basyang_TCA#01                TL    702     4.7m         -48
22-TC02_Basyang_TCA#01               CEB    662     4.4m         -88
------------------------------------------------------------
Target: 750 words ≈ 5 minutes at 150 wpm

✓ 6 scripts saved to: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins
